# Linear Slim ffill — Mon open → Fri close target (S1 Equities extras)

- This is better than a median approach

Pooled **Ridge** on engineered features after an editable **`FEATURE_REMOVE`** exclude-list
(same list / structure as `04_training_linear_slim`).
Target = within-date pct rank of **`fwd_ret_mon_fri`** (Monday open → Friday close) on week-start rows.

This extras notebook mirrors `05_training_linear_slim_ffill` but changes only the Ridge label so timing bake-offs vs Mon→Mon are not label-biased.
Artifacts are written under `model_tests/extras/` (not `model_artifacts/`).

NaN policy: within-ticker **forward-fill** of raw feature values to the most recent
prior observation, then CS-rank, then **drop any row** still NaN on a model feature
(complete-case), with drop counts printed.

## Ridge vs ordinary least squares (OLS)

OLS fits coefficients by minimizing squared prediction error only. With several
cross-sectionally ranked factors, those coefficients can become large and unstable
when features are correlated.

**Ridge** is the same linear model with an L2 penalty: it minimizes
`||y − Xβ||² + α||β||²`. The penalty shrinks coefficients toward zero. This notebook
uses Ridge and chooses `α` on the sealed validation window by mean date IC.


## 0. Imports & Config


In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from models.s1_equities.training_common import (
    LABEL_COL_MON_FRI as LABEL_COL,
    TARGET_COL_MON_FRI as TARGET_COL,
    add_cs_pct_target,
    attach_scores,
    chronological_is_split,
    cs_rank_features,
    default_paths,
    drop_nonfinite_labels,
    ic_segment_table,
    mean_date_ic,
    prepare_s1_week_panel,
)

from data.processing.cleaner import forward_fill_panel

PATHS = default_paths(ROOT)
FEATURES_PATH = PATHS["features"]
TRAIN_PANEL_PATH = PATHS["train_panel"]
EXTRAS_DIR = os.path.join(
    ROOT, "03_models", "s1_equities", "model_tests", "extras"
)
os.makedirs(EXTRAS_DIR, exist_ok=True)
PRED_PATH = os.path.join(
    EXTRAS_DIR, "s1_linear_slim_ffill_mon_fri_is_predictions.parquet"
)
MODEL_PATH = os.path.join(
    EXTRAS_DIR, "s1_linear_slim_ffill_mon_fri.joblib"
)

VAL_FRAC = 0.15
EMBARGO_WEEKS = 1
ALPHA_GRID = [0.1, 1.0, 10.0, 100.0]
RANDOM_SEED = 42

# Editable remove-list (screening names mapped to engineered columns where needed).
# Comment / uncomment lines to add or remove factors from the *exclude* set.
# Names missing from the panel are skipped with a warning.
FEATURE_REMOVE = [
    #"log_mcap",
    #"val_mom_dist_252_21",
    #"downside_beta_42",
    #"filing_clock_expected_until",
    "earnings_yield",
    "smart_beta_mom_126",
    "gdelt_abnormal_attention_5_60",
    "smart_beta_smb_84",
    "val_mom_resid_126_252_10",
    #"upside_beta_42",
    #"gdelt_attention_5",
    "obv_mom_soft_126_21_20",
    #"smart_residual_mom_189_42",  # screening: smart_residual_mom_189_21
    #"val_roc_pe_252",
    #"rel_upside_beta_63",
    #"gross_profitability",
    "near_52w_ratio_252_raw",
]

print(f"ROOT={ROOT}")
print(f"FEATURES_PATH={FEATURES_PATH}")
print(f"PRED_PATH={PRED_PATH}")
print(f"FEATURE_REMOVE requested ({len(FEATURE_REMOVE)}): {FEATURE_REMOVE}")

ROOT=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
FEATURES_PATH=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_engineered_features.parquet
PRED_PATH=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_tests\extras\s1_linear_slim_ffill_mon_fri_is_predictions.parquet
FEATURE_REMOVE requested (7): ['earnings_yield', 'smart_beta_mom_126', 'gdelt_abnormal_attention_5_60', 'smart_beta_smb_84', 'val_mom_resid_126_252_10', 'obv_mom_soft_126_21_20', 'near_52w_ratio_252_raw']


## 1. Load week-start panel

Uses `prepare_s1_week_panel`, then drops `FEATURE_REMOVE` names from the full feature set.


In [2]:
df, all_feats, _prices = prepare_s1_week_panel(
    FEATURES_PATH, TRAIN_PANEL_PATH, attach_mon_fri_label=True
)

missing = [c for c in FEATURE_REMOVE if c not in df.columns]
remove_resolved = [c for c in FEATURE_REMOVE if c in df.columns]
if missing:
    print(f"WARNING: {len(missing)} remove-list name(s) not in panel (skipped): {missing}")

FEATURE_COLS = [c for c in all_feats if c not in set(remove_resolved)]
print(f"FEATURE_REMOVE resolved ({len(remove_resolved)}): {remove_resolved}")
print(f"FEATURE_COLS kept ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Week-start rows: {df.shape}  unique weeks={df['date'].nunique():,}")
print(
    f"IS weeks={df.loc[df['is_research_is'], 'date'].nunique():,}  "
    f"holdout weeks={df.loc[~df['is_research_is'], 'date'].nunique():,}"
)
df.head()

FEATURE_REMOVE resolved (7): ['earnings_yield', 'smart_beta_mom_126', 'gdelt_abnormal_attention_5_60', 'smart_beta_smb_84', 'val_mom_resid_126_252_10', 'obv_mom_soft_126_21_20', 'near_52w_ratio_252_raw']
FEATURE_COLS kept (20): ['raw_momentum_252_5', 'smart_residual_mom_189_42', 'rel_downside_beta_252', 'rel_upside_beta_63', 'smart_beta_hml_252', 'downside_beta_42', 'upside_beta_42', 'size_mom_126', 'val_roc_pb_252', 'val_roc_pe_252', 'log_mcap', 'val_mom_dist_252_21', 'gross_profitability', 'filing_clock_expected_until', 'short_flow_ratio', 'market_corr', 'beta_mkt_interact', 'abnormal_volume', 'gdelt_tone_x_attention_21', 'gdelt_attention_5']
Week-start rows: (85279, 40)  unique weeks=865
IS weeks=422  holdout weeks=443


,date,ticker,feature_date,open,high,low,close,volume,fwd_ret_1,fwd_ret_5,...,filing_clock_expected_until,short_flow_ratio,market_corr,beta_mkt_interact,abnormal_volume,gdelt_tone_x_attention_21,gdelt_attention_5,gdelt_abnormal_attention_5_60,fwd_ret_mon_fri,is_research_is
0,2010-01-05,AAPL,2010-01-04,6.424143,6.421146,6.357683,6.406478,493729600.0,-0.001025,-0.025210,...,27.0,0.439729,NaN,NaN,NaN,NaN,NaN,NaN,-0.012209,False
1,2010-01-05,ABT,2010-01-04,18.082116,18.111994,17.899535,18.078796,10829095.0,-0.009730,0.013402,...,31.0,0.255644,NaN,NaN,NaN,NaN,NaN,NaN,0.010649,False
2,2010-01-05,ADBE,2010-01-04,37.040001,37.299999,36.650002,37.090000,4710200.0,0.007829,-0.024298,...,1.0,0.383420,NaN,NaN,NaN,NaN,NaN,NaN,-0.009449,False
3,2010-01-05,AET,2010-01-04,29.889494,30.016532,28.918587,29.943939,5671979.0,-0.013358,-0.009411,...,NaN,0.295322,NaN,NaN,NaN,NaN,NaN,NaN,-0.007286,False
4,2010-01-05,AIG,2010-01-04,18.609565,18.957173,18.255744,18.553696,7750900.0,-0.021014,-0.013009,...,4.0,0.417434,NaN,NaN,NaN,NaN,NaN,NaN,-0.021348,False


## 2. Forward-fill raw features

Within-ticker ffill on model features (unlimited): each NaN becomes the most recent
prior non-null value for that ticker on the week-start panel.


In [3]:
_cov_before = {c: float(df[c].notna().mean()) for c in FEATURE_COLS}
df = forward_fill_panel(df, columns=FEATURE_COLS, limit=None)
print("FFILL coverage on FEATURE_COLS (week-start, before -> after):")
for c in FEATURE_COLS:
    after = float(df[c].notna().mean())
    delta = after - _cov_before[c]
    if delta > 1e-6 or _cov_before[c] < 0.999:
        print(
            f"  {c}: {100 * _cov_before[c]:.2f}% -> {100 * after:.2f}% "
            f"({100 * delta:+.2f} pp)"
        )

FFILL coverage on FEATURE_COLS (week-start, before -> after):
  raw_momentum_252_5: 93.79% -> 93.79% (+0.00 pp)
  smart_residual_mom_189_42: 70.68% -> 70.68% (+0.00 pp)
  rel_downside_beta_252: 93.79% -> 93.79% (+0.00 pp)
  rel_upside_beta_63: 98.36% -> 98.36% (+0.00 pp)
  smart_beta_hml_252: 73.73% -> 73.73% (+0.00 pp)
  downside_beta_42: 68.86% -> 98.59% (+29.73 pp)
  upside_beta_42: 82.29% -> 98.83% (+16.54 pp)
  size_mom_126: 84.77% -> 88.80% (+4.03 pp)
  val_roc_pb_252: 71.06% -> 81.69% (+10.63 pp)
  val_roc_pe_252: 62.09% -> 87.71% (+25.62 pp)
  log_mcap: 88.25% -> 94.68% (+6.43 pp)
  val_mom_dist_252_21: 73.52% -> 83.67% (+10.15 pp)
  gross_profitability: 60.09% -> 60.09% (+0.00 pp)
  filing_clock_expected_until: 65.30% -> 94.26% (+28.96 pp)
  short_flow_ratio: 69.61% -> 99.18% (+29.57 pp)
  market_corr: 93.79% -> 93.79% (+0.00 pp)
  beta_mkt_interact: 93.79% -> 93.79% (+0.00 pp)
  abnormal_volume: 98.36% -> 98.36% (+0.00 pp)
  gdelt_tone_x_attention_21: 57.12% -> 63.33% (+6.21 

## 3. CS-rank target

Within-date pct rank of `fwd_ret_mon_fri` (Mon open → Fri close).


In [4]:
df = add_cs_pct_target(df, label_col=LABEL_COL, target_col=TARGET_COL)
n_before = len(df)
df = drop_nonfinite_labels(df, [LABEL_COL, TARGET_COL])
print(
    f"Rows after finite label/target: {df.shape}  "
    f"(retained {100 * len(df) / max(n_before, 1):.2f}%)"
)
print(
    f"Target mean={df[TARGET_COL].mean():.4f}  std={df[TARGET_COL].std():.4f}"
)

Rows after finite label/target: (85140, 41)  (retained 99.84%)
Target mean=0.5051  std=0.2887


## 4. CS-rank features + drop remaining NaNs

CS-rank after ffill. Drop any row with NaN still present in a model feature.


In [5]:
X_ranked = cs_rank_features(df, FEATURE_COLS, fill_value=None)

nan_share = X_ranked.isna().mean().sort_values(ascending=False)
print("Per-feature NaN share after ffill + CS-rank (before drop):")
display((100 * nan_share).rename("nan_pct").to_frame().round(2).head(20))

n_before = len(df)
dates_before = df["date"].nunique()
span_before = (df["date"].min(), df["date"].max())
nan_any = X_ranked.isna().any(axis=1)
n_dropped = int(nan_any.sum())
df = df.loc[~nan_any].copy()
X = X_ranked.loc[~nan_any].copy()
n_after = len(df)

print(
    "Complete-case drop (NaN in any model feature after ffill):\n"
    f"  rows:  before={n_before:,}  after={n_after:,}  "
    f"dropped={n_dropped:,} ({100 * n_dropped / max(n_before, 1):.2f}%)\n"
    f"  dates: before={dates_before:,}  after={df['date'].nunique():,}\n"
    f"  span:  before={span_before[0].date()} -> {span_before[1].date()}  "
    f"after={(df['date'].min().date() if n_after else 'n/a')} -> "
    f"{(df['date'].max().date() if n_after else 'n/a')}"
)
if n_after == 0:
    raise ValueError(
        "No rows left after complete-case drop; relax FEATURE_REMOVE or ffill coverage."
    )

is_end_probe = df.loc[df["is_research_is"], "date"].max()
post_is = df.loc[(~df["is_research_is"]) & (df["date"] > is_end_probe)]
if post_is.empty:
    print(
        "WARNING: no post-IS holdout rows remain after complete-case drop. "
        "OS holdout IC will be undefined until coverage improves."
    )

Per-feature NaN share after ffill + CS-rank (before drop):


,nan_pct
gross_profitability,39.93
gdelt_tone_x_attention_21,36.71
gdelt_attention_5,34.21
smart_residual_mom_189_42,29.36
smart_beta_hml_252,26.31
val_roc_pb_252,18.33
val_mom_dist_252_21,16.35
val_roc_pe_252,12.30
size_mom_126,11.21
beta_mkt_interact,6.23


Complete-case drop (NaN in any model feature after ffill):
  rows:  before=85,140  after=35,062  dropped=50,078 (58.82%)
  dates: before=864  after=592
  span:  before=2010-01-05 -> 2026-07-20  after=2015-03-23 -> 2026-07-20


## 5. Train / Val / Holdout Split


In [6]:
split = chronological_is_split(df, val_frac=VAL_FRAC, embargo_weeks=EMBARGO_WEEKS)
train_dates, val_dates = split.train_dates, split.val_dates
is_end = split.is_end
train_df, val_df, holdout_df = split.train_df, split.val_df, split.holdout_df

for label, part in (
    ("train", train_df),
    ("val", val_df),
    ("holdout", holdout_df),
):
    print(f"{label}: rows={len(part):,}  weeks={part['date'].nunique()}")

print(
    f"Train:   {train_df.shape}  {train_dates.min().date()} -> {train_dates.max().date()}  "
    f"({len(train_dates)} weeks)"
)
print(f"Embargo: {list(split.embargo_dates.date)}  ({len(split.embargo_dates)} weeks)")
print(
    f"Val:     {val_df.shape}  {val_dates.min().date()} -> {val_dates.max().date()}  "
    f"({len(val_dates)} weeks)"
)
print(
    f"Holdout: {holdout_df.shape}  "
    f"{holdout_df['date'].min().date() if len(holdout_df) else 'n/a'} -> "
    f"{holdout_df['date'].max().date() if len(holdout_df) else 'n/a'}  "
    f"({holdout_df['date'].nunique()} weeks)"
)

train: rows=18,827  weeks=348
val: rows=4,030  weeks=62
holdout: rows=12,140  weeks=181
Train:   (18827, 41)  2015-03-23 -> 2021-11-15  (348 weeks)
Embargo: [datetime.date(2021, 11, 22)]  (1 weeks)
Val:     (4030, 41)  2021-11-29 -> 2023-01-30  (62 weeks)
Holdout: (12140, 41)  2023-02-06 -> 2026-07-20  (181 weeks)


## 6. Fit Ridge

`α` selected on val by mean date IC. Features are CS-ranked and complete-case.


In [7]:
X_train = X.loc[train_df.index]
y_train = train_df[TARGET_COL]
X_val = X.loc[val_df.index]

search_rows = []
for alpha in ALPHA_GRID:
    model = Ridge(alpha=alpha, random_state=RANDOM_SEED)
    model.fit(X_train, y_train)
    pred_va = attach_scores(val_df, model.predict(X_val), label_col=LABEL_COL)
    sm = mean_date_ic(pred_va, label_col=LABEL_COL)
    search_rows.append(
        {
            "alpha": alpha,
            "val_mean_ic": sm["mean_ic"],
            "val_icir": sm["icir"],
            "n_dates": sm["n"],
        }
    )
    print(
        f"alpha={alpha:<6g}  val_ic={sm['mean_ic']:.4f}  "
        f"ICIR={sm['icir']:.3f}  n={sm['n']}"
    )

search_df = (
    pd.DataFrame(search_rows)
    .sort_values(["val_mean_ic", "val_icir"], ascending=False)
    .reset_index(drop=True)
)
display(search_df)

BEST_ALPHA = float(search_df.iloc[0]["alpha"])
print(f"BEST_ALPHA={BEST_ALPHA}  (val mean IC={search_df.iloc[0]['val_mean_ic']:.4f})")

alpha=0.1     val_ic=-0.0369  ICIR=-0.107  n=62
alpha=1       val_ic=-0.0369  ICIR=-0.107  n=62
alpha=10      val_ic=-0.0371  ICIR=-0.107  n=62
alpha=100     val_ic=-0.0380  ICIR=-0.110  n=62


,alpha,val_mean_ic,val_icir,n_dates
0,1.0,-0.036875,-0.106750,62
1,0.1,-0.036925,-0.106832,62
2,10.0,-0.037109,-0.107348,62
3,100.0,-0.038046,-0.110037,62


BEST_ALPHA=1.0  (val mean IC=-0.0369)


## 7. Final IS fit & predictions


In [9]:
model = Ridge(alpha=BEST_ALPHA, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

preds = attach_scores(df, model.predict(X), label_col=LABEL_COL)
if "feature_date" in df.columns:
    preds = preds.merge(
        df[["date", "ticker", "feature_date"]],
        on=["date", "ticker"],
        how="left",
    )
preds.to_parquet(PRED_PATH, index=False)
import joblib
joblib.dump({"model": model, "feature_cols": FEATURE_COLS, "alpha": BEST_ALPHA}, MODEL_PATH)
print(f"Saved predictions: {PRED_PATH}")
print(f"Saved model: {MODEL_PATH}")
print(
    f"  rows={len(preds):,}  IS={preds['is_research_is'].sum():,}  "
    f"non-IS={(~preds['is_research_is']).sum():,}"
)

coef = (
    pd.Series(model.coef_, index=FEATURE_COLS)
    .sort_values(key=np.abs, ascending=False)
    .rename("coef")
)
display(coef.head(20).to_frame())

Saved predictions: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_tests\extras\s1_linear_slim_ffill_mon_fri_is_predictions.parquet
Saved model: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_tests\extras\s1_linear_slim_ffill_mon_fri.joblib
  rows=35,062  IS=22,922  non-IS=12,140


,coef
smart_beta_hml_252,-0.044585
market_corr,0.022832
gdelt_attention_5,0.021757
smart_residual_mom_189_42,0.020866
gdelt_tone_x_attention_21,0.020326
gross_profitability,0.017683
abnormal_volume,-0.017528
log_mcap,-0.016868
val_mom_dist_252_21,0.013054
raw_momentum_252_5,-0.012350


## 8. IC summary


In [10]:
ic_compare = ic_segment_table(preds, train_dates, val_dates, is_end, label_col=LABEL_COL)
display(ic_compare)

ho = ic_compare.loc["OS holdout"]
tv = ic_compare.loc["IS train+val"]
print(
    f"IS train+val IC={tv['mean_ic']:.4f} ICIR={tv['icir']:.3f}  |  "
    f"OS holdout IC={ho['mean_ic']:.4f} ICIR={ho['icir']:.3f}"
)

,mean_ic,icir,n_dates
segment,,,
IS train,0.061468,0.285815,348
IS val,-0.036875,-0.106750,62
IS train+val,0.046597,0.193016,410
OS holdout,0.024717,0.096560,181


IS train+val IC=0.0466 ICIR=0.193  |  OS holdout IC=0.0247 ICIR=0.097
